# GPU Experiment: Uncertainty Quantification Methods Comparison

**Dataset:** FB15k-237 (14,541 entities, 237 relations)

**Models:**
1. DistMult (baseline)
2. DistMult + MC Dropout
3. GGPN (Graph Gaussian Process Network)
4. GP-KGE (Our method)

**Metrics:**
- Link Prediction: MRR, Hits@1, Hits@3, Hits@10
- Calibration: ECE (Expected Calibration Error), Brier Score
- OOD Detection: AUROC

---
## Stage 1: Environment Setup
---

In [ ]:
# %cd /content/kg-bayesian-prior
# !git pull

/content/kg-bayesian-prior
Already up to date.


In [14]:
# ============================================================
# STAGE 1: ENVIRONMENT SETUP
# ============================================================
import sys
import os
from datetime import datetime

def log_stage(stage_name):
    """Print formatted stage header with timestamp."""
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"\n{'='*60}")
    print(f"[{timestamp}] {stage_name}")
    print(f"{'='*60}")

def log_step(step_name):
    """Print formatted step with timestamp."""
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] >>> {step_name}")

def log_done(message="Done"):
    """Print completion message."""
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] ✓ {message}")

log_stage("STAGE 1: ENVIRONMENT SETUP")

# Detect Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    log_step("Cloning repository...")
    !git clone https://github.com/ChorokLeeDev/kg-bayesian-prior.git /content/kg-bayesian-prior 2>/dev/null || echo "Already cloned"
    %cd /content/kg-bayesian-prior

    log_step("Installing dependencies...")
    !pip install -q torch-geometric gpytorch pykeen networkx pandas tqdm scikit-learn matplotlib seaborn

log_done("Environment setup complete")


[06:47:41] STAGE 1: ENVIRONMENT SETUP
Environment: Google Colab
[06:47:41] >>> Cloning repository...
Already cloned
/content/kg-bayesian-prior
[06:47:41] >>> Installing dependencies...
[06:47:47] ✓ Environment setup complete


In [15]:
# Setup Python paths
from pathlib import Path

if IN_COLAB:
    ROOT_DIR = Path('/content/kg-bayesian-prior')
else:
    ROOT_DIR = Path('..').resolve()

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
if str(ROOT_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT_DIR / 'src'))

print(f"Root: {ROOT_DIR}")

Root: /content/kg-bayesian-prior


In [16]:
# Check GPU
import torch

log_step("Checking GPU availability...")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    log_done("GPU available")
else:
    print("⚠️ WARNING: No GPU detected!")
    print("Go to: Runtime -> Change runtime type -> GPU")

[06:47:47] >>> Checking GPU availability...
Device: cuda
GPU: Tesla T4
Memory: 15.8 GB
[06:47:47] ✓ GPU available


---
## Stage 2: Load Data & Imports
---

In [17]:
# ============================================================
# STAGE 2: IMPORTS AND DATA LOADING
# ============================================================
log_stage("STAGE 2: IMPORTS AND DATA LOADING")

import json
import gc
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

log_step("Importing project modules...")
from src.data import load_fb15k237
from src.models import DistMult, GPKGE
from src.models.ggpn import GGPN
from src.models.uncertain_kge import MCDropoutKGE
from src.utils.training import set_seed, NegativeSampler
from src.evaluation.calibration import expected_calibration_error, brier_score
from src.evaluation.ood_detection import compute_auroc, create_ood_dataset

log_done("Imports complete")


[06:47:47] STAGE 2: IMPORTS AND DATA LOADING
[06:47:47] >>> Importing project modules...
[06:47:47] ✓ Imports complete


In [18]:
# Load FB15k-237
log_step("Loading FB15k-237 dataset...")

set_seed(42)
train_data, valid_data, test_data = load_fb15k237()

print(f"\n  Dataset Statistics:")
print(f"  - Entities:  {train_data.num_entities:,}")
print(f"  - Relations: {train_data.num_relations}")
print(f"  - Train:     {len(train_data):,} triples")
print(f"  - Valid:     {len(valid_data):,} triples")
print(f"  - Test:      {len(test_data):,} triples")

log_done("Data loaded")

[06:47:47] >>> Loading FB15k-237 dataset...

  Dataset Statistics:
  - Entities:  14,541
  - Relations: 237
  - Train:     272,115 triples
  - Valid:     17,535 triples
  - Test:      20,466 triples
[06:47:51] ✓ Data loaded


---
## Stage 3: Define Training & Evaluation Functions
---

In [19]:
# ============================================================
# STAGE 3: TRAINING & EVALUATION FUNCTIONS
# ============================================================
log_stage("STAGE 3: DEFINE FUNCTIONS")

def train_model(model, train_data, num_epochs=50, batch_size=1024, lr=0.001,
                device="cuda", model_name="Model"):
    """Train model with progress tracking."""
    log_step(f"Training {model_name}...")

    model = model.to(device)

    # Set graph for GP-based models
    if hasattr(model, 'set_graph'):
        log_step(f"Setting graph structure for {model_name}...")
        model.set_graph(train_data)
        log_done("Graph structure set")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    neg_sampler = NegativeSampler(train_data.num_entities, num_negatives=10)
    neg_sampler.set_true_triples(train_data.triples)

    # Training loop with progress bar
    pbar = tqdm(range(num_epochs), desc=f"Training {model_name}", unit="epoch")

    for epoch in pbar:
        model.train()
        total_loss = 0
        num_batches = 0
        indices = np.random.permutation(len(train_data))

        for start in range(0, len(indices), batch_size):
            end = min(start + batch_size, len(indices))
            batch = train_data.triples[indices[start:end]]

            pos_triples = torch.tensor(batch, device=device)
            neg_triples = neg_sampler(pos_triples).to(device)

            optimizer.zero_grad()

            if hasattr(model, 'loss'):
                loss_dict = model.loss(pos_triples, neg_triples)
                loss = loss_dict['total'] if isinstance(loss_dict, dict) else loss_dict
            else:
                pos_scores = model(pos_triples[:, 0], pos_triples[:, 1], pos_triples[:, 2])
                neg_scores = model(neg_triples[:, 0], neg_triples[:, 1], neg_triples[:, 2])
                num_neg = len(neg_triples) // len(pos_triples)
                if num_neg > 1:
                    neg_scores = neg_scores.view(len(pos_triples), num_neg).mean(dim=1)
                loss = F.relu(1.0 - pos_scores + neg_scores).mean()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        avg_loss = total_loss / num_batches
        pbar.set_postfix({"loss": f"{avg_loss:.4f}"})

    log_done(f"{model_name} training complete")
    return model


def evaluate_model(model, test_data, train_data, device, model_name="Model", num_samples=2000):
    """Full evaluation with progress tracking."""
    log_step(f"Evaluating {model_name}...")
    model.eval()

    # === Link Prediction ===
    log_step("Computing link prediction metrics (MRR, Hits@k)...")
    sample_idx = np.random.choice(len(test_data), min(num_samples, len(test_data)), replace=False)
    sample = test_data.triples[sample_idx]

    all_ranks = []
    batch_size = 200

    with torch.no_grad():
        for start in tqdm(range(0, len(sample), batch_size), desc="Link Prediction", leave=False):
            end = min(start + batch_size, len(sample))
            batch = sample[start:end]

            h = torch.tensor(batch[:, 0], device=device)
            r = torch.tensor(batch[:, 1], device=device)
            t = torch.tensor(batch[:, 2], device=device)

            if hasattr(model, 'score_tails'):
                scores = model.score_tails(h, r)
            else:
                all_tails = torch.arange(test_data.num_entities, device=device)
                scores = []
                for i in range(len(h)):
                    s = model(h[i].expand(test_data.num_entities), r[i].expand(test_data.num_entities), all_tails)
                    scores.append(s)
                scores = torch.stack(scores)

            target_scores = scores[torch.arange(len(t), device=device), t]
            ranks = (scores > target_scores.unsqueeze(1)).sum(dim=1) + 1
            all_ranks.extend(ranks.cpu().tolist())

    ranks = torch.tensor(all_ranks, dtype=torch.float)
    mrr = (1.0 / ranks).mean().item()
    hits1 = (ranks <= 1).float().mean().item()
    hits3 = (ranks <= 3).float().mean().item()
    hits10 = (ranks <= 10).float().mean().item()

    # === Calibration ===
    log_step("Computing calibration metrics (ECE, Brier)...")
    pos_idx = np.random.choice(len(test_data), min(1000, len(test_data)), replace=False)
    pos_triples = test_data.triples[pos_idx]
    neg_triples = np.array([[h, r, np.random.randint(test_data.num_entities)] for h, r, t in pos_triples])

    all_triples = np.vstack([pos_triples, neg_triples])
    labels = np.concatenate([np.ones(len(pos_triples)), np.zeros(len(neg_triples))])

    shuffle_idx = np.random.permutation(len(all_triples))
    all_triples = all_triples[shuffle_idx]
    labels = labels[shuffle_idx]

    with torch.no_grad():
        h = torch.tensor(all_triples[:, 0], device=device)
        r = torch.tensor(all_triples[:, 1], device=device)
        t = torch.tensor(all_triples[:, 2], device=device)

        if hasattr(model, 'base_model'):
            scores = model.base_model.score_triple(h, r, t)
        elif hasattr(model, 'score_triple'):
            scores = model.score_triple(h, r, t)
        else:
            scores = model(h, r, t)

        confidences = torch.sigmoid(scores).cpu().numpy()

    ece, _ = expected_calibration_error(confidences, labels)
    brier = brier_score(confidences, labels)

    # === OOD Detection ===
    log_step("Computing OOD detection (AUROC)...")
    id_triples = test_data.triples[np.random.choice(len(test_data), min(1000, len(test_data)), replace=False)]
    ood_triples = create_ood_dataset(train_data, test_data, "random", 1000)

    def get_uncertainty(triples):
        h = torch.tensor(triples[:, 0], device=device)
        r = torch.tensor(triples[:, 1], device=device)
        t = torch.tensor(triples[:, 2], device=device)

        with torch.no_grad():
            if hasattr(model, 'predict_with_uncertainty'):
                pred = model.predict_with_uncertainty(h, r, t)
                if isinstance(pred, dict):
                    return pred.get('total', pred.get('epistemic', torch.zeros(len(h)))).cpu().numpy()
                return pred[1].cpu().numpy()
            elif hasattr(model, 'predict_with_mc_samples'):
                _, var = model.predict_with_mc_samples(h, r, t, num_samples=10)
                return var.cpu().numpy()
            else:
                if hasattr(model, 'score_triple'):
                    scores = model.score_triple(h, r, t)
                else:
                    scores = model(h, r, t)
                probs = torch.sigmoid(scores)
                entropy = -probs * torch.log(probs + 1e-10) - (1-probs) * torch.log(1-probs + 1e-10)
                return entropy.cpu().numpy()

    auroc = compute_auroc(get_uncertainty(id_triples), get_uncertainty(ood_triples))

    results = {
        "mrr": mrr, "hits@1": hits1, "hits@3": hits3, "hits@10": hits10,
        "ece": ece, "brier": brier, "auroc": auroc
    }

    # Print results
    print(f"\n  {model_name} Results:")
    print(f"  Link Prediction: MRR={mrr:.4f}, H@1={hits1:.4f}, H@3={hits3:.4f}, H@10={hits10:.4f}")
    print(f"  Calibration:     ECE={ece:.4f}, Brier={brier:.4f}")
    print(f"  OOD Detection:   AUROC={auroc:.4f}")

    log_done(f"{model_name} evaluation complete")
    return results


def clear_memory():
    """Clear GPU memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


log_done("Functions defined")


[06:47:51] STAGE 3: DEFINE FUNCTIONS
[06:47:51] ✓ Functions defined


---
## Stage 4: Run Experiments
---

In [20]:
# ============================================================
# STAGE 4: RUN EXPERIMENTS
# ============================================================
log_stage("STAGE 4: RUN EXPERIMENTS")

# Store all results
results = {}

print("\nModels to train:")
print("  1. DistMult (baseline)")
print("  2. DistMult + MC Dropout")
print("  3. GGPN (reduced params for T4 GPU)")
print("  4. GP-KGE (our method)")
print("\nEstimated total time: 15-25 minutes on T4 GPU")


[06:47:51] STAGE 4: RUN EXPERIMENTS

Models to train:
  1. DistMult (baseline)
  2. DistMult + MC Dropout
  3. GGPN (reduced params for T4 GPU)
  4. GP-KGE (our method)

Estimated total time: 15-25 minutes on T4 GPU


### 4.1 DistMult (Baseline)

In [ ]:
# --- Model 1: DistMult ---
log_stage("MODEL 1/4: DistMult (Baseline)")
clear_memory()

distmult = DistMult(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=200
)

distmult = train_model(distmult, train_data, num_epochs=50, device=device, model_name="DistMult")
results["DistMult"] = evaluate_model(distmult, test_data, train_data, device, "DistMult")

del distmult
clear_memory()


[06:47:51] MODEL 1/4: DistMult (Baseline)
[06:47:52] >>> Training DistMult...


Training DistMult:   0%|          | 0/50 [00:00<?, ?epoch/s]

### 4.2 DistMult + MC Dropout

In [ ]:
# --- Model 2: MC Dropout ---
log_stage("MODEL 2/4: DistMult + MC Dropout")
clear_memory()

base_model = DistMult(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=200,
    dropout=0.3
)
mc_dropout = MCDropoutKGE(base_model, num_samples=20)

mc_dropout = train_model(mc_dropout, train_data, num_epochs=50, device=device, model_name="MCDropout")
results["DistMult+MCDropout"] = evaluate_model(mc_dropout, test_data, train_data, device, "MCDropout")

del mc_dropout, base_model
clear_memory()

### 4.3 GGPN (Reduced Parameters for T4)

In [ ]:
# --- Model 3: GGPN ---
log_stage("MODEL 3/4: GGPN (Graph Gaussian Process Network)")
clear_memory()

print("Note: Using reduced parameters to fit T4 GPU memory (15GB)")
print("  - embedding_dim: 50 (vs 200)")
print("  - hidden_dim: 50 (vs 200)")
print("  - num_layers: 1 (vs 2)")
print("  - num_rff: 20 (vs 200)")

ggpn = GGPN(
    train_data.num_entities,
    train_data.num_relations * 2,  # Forward + backward edges
    embedding_dim=50,
    hidden_dim=50,
    num_layers=1,
    num_rff=20,
)

ggpn = train_model(ggpn, train_data, num_epochs=50, batch_size=512, device=device, model_name="GGPN")
results["GGPN"] = evaluate_model(ggpn, test_data, train_data, device, "GGPN")

del ggpn
clear_memory()

### 4.4 GP-KGE (Our Method)

In [ ]:
# --- Model 4: GP-KGE ---
log_stage("MODEL 4/4: GP-KGE (Our Method)")
clear_memory()

print("Using RELATION-AWARE kernel (our main contribution)")
print("Optimizations for T4 GPU:")
print("  - num_eigenvectors: 100 (vs 1000)")
print("  - min_edges: 10 (skip sparse relations)")
print("  - Progress bar for eigendecomposition")

gpkge = GPKGE(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=200,
    kernel_type="relation_aware",  # OUR MAIN CONTRIBUTION
    scoring_function="distmult",
    num_inducing=min(500, train_data.num_entities),
)

# Custom training to pass eigendecomp params
log_step("Training GP-KGE...")
gpkge = gpkge.to(device)

log_step("Setting graph structure (eigendecomposition)...")
gpkge.set_graph(train_data, num_eigenvectors=100, min_edges=10, show_progress=True)
log_done("Graph structure set")

optimizer = torch.optim.Adam(gpkge.parameters(), lr=0.001)
neg_sampler = NegativeSampler(train_data.num_entities, num_negatives=10)
neg_sampler.set_true_triples(train_data.triples)

pbar = tqdm(range(50), desc="Training GP-KGE", unit="epoch")
for epoch in pbar:
    gpkge.train()
    total_loss = 0
    num_batches = 0
    indices = np.random.permutation(len(train_data))
    
    for start in range(0, len(indices), 1024):
        end = min(start + 1024, len(indices))
        batch = train_data.triples[indices[start:end]]
        
        pos_triples = torch.tensor(batch, device=device)
        neg_triples = neg_sampler(pos_triples).to(device)
        
        optimizer.zero_grad()
        loss_dict = gpkge.loss(pos_triples, neg_triples)
        loss = loss_dict['total']
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gpkge.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    pbar.set_postfix({"loss": f"{total_loss/num_batches:.4f}"})

log_done("GP-KGE training complete")
results["GP-KGE (Ours)"] = evaluate_model(gpkge, test_data, train_data, device, "GP-KGE")

del gpkge
clear_memory()

---
## Stage 5: Results Summary
---

In [ ]:
# ============================================================
# STAGE 5: RESULTS SUMMARY
# ============================================================
log_stage("STAGE 5: RESULTS SUMMARY")

print(f"\n{'Model':<20} {'MRR':>8} {'H@1':>8} {'H@10':>8} {'ECE':>8} {'Brier':>8} {'AUROC':>8}")
print("-" * 80)
for name, r in results.items():
    print(f"{name:<20} {r['mrr']:>8.4f} {r['hits@1']:>8.4f} {r['hits@10']:>8.4f} "
          f"{r['ece']:>8.4f} {r['brier']:>8.4f} {r['auroc']:>8.4f}")

# Highlight key findings
if "GGPN" in results and "GP-KGE (Ours)" in results:
    ggpn_ece = results["GGPN"]["ece"]
    gpkge_ece = results["GP-KGE (Ours)"]["ece"]
    improvement = (ggpn_ece - gpkge_ece) / ggpn_ece * 100 if ggpn_ece > 0 else 0

    print(f"\n{'='*60}")
    print("KEY FINDING: Calibration Comparison")
    print(f"{'='*60}")
    print(f"  GGPN ECE:     {ggpn_ece:.4f}")
    print(f"  GP-KGE ECE:   {gpkge_ece:.4f}")
    print(f"  Improvement:  {improvement:.1f}%")

log_done("Experiment complete!")

In [ ]:
# Visualize results
log_step("Generating visualization...")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

model_names = list(results.keys())
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# Link Prediction (MRR)
mrrs = [results[m]['mrr'] for m in model_names]
axes[0].bar(model_names, mrrs, color=colors[:len(model_names)])
axes[0].set_ylabel('MRR')
axes[0].set_title('Link Prediction (MRR) - Higher is Better')
axes[0].tick_params(axis='x', rotation=45)

# Calibration (ECE - lower is better)
eces = [results[m]['ece'] for m in model_names]
axes[1].bar(model_names, eces, color=colors[:len(model_names)])
axes[1].set_ylabel('ECE')
axes[1].set_title('Calibration Error (ECE) - Lower is Better')
axes[1].tick_params(axis='x', rotation=45)

# OOD Detection (AUROC - higher is better)
aurocs = [results[m]['auroc'] for m in model_names]
axes[2].bar(model_names, aurocs, color=colors[:len(model_names)])
axes[2].set_ylabel('AUROC')
axes[2].set_title('OOD Detection (AUROC) - Higher is Better')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(ROOT_DIR / 'outputs' / 'results_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

log_done("Visualization saved")

---
## Stage 6: Save Results
---

In [ ]:
# ============================================================
# STAGE 6: SAVE RESULTS
# ============================================================
log_stage("STAGE 6: SAVE RESULTS")

# Create output directory
save_dir = ROOT_DIR / "outputs" / "fb15k237"
save_dir.mkdir(parents=True, exist_ok=True)

# Convert numpy types to Python types for JSON
def to_serializable(obj):
    if isinstance(obj, (np.floating, np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

# Save to JSON
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_path = save_dir / f"gpu_results_{timestamp}.json"

serializable_results = {
    k: {kk: to_serializable(vv) for kk, vv in v.items()}
    for k, v in results.items()
}

with open(results_path, "w") as f:
    json.dump(serializable_results, f, indent=2)

print(f"Results saved to: {results_path}")

# Download results if on Colab
if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(results_path))
        log_done("Results downloaded")
    except:
        print("Note: Download widget not available in VS Code extension")
        print(f"Results saved at: {results_path}")

log_done("All done!")

---
## Utility: Re-clone & Restart

Run this cell if you need to pull latest code changes:
---

In [ ]:
# # Unassign Colab runtime before exiting

# from google.colab import runtime
# runtime.unassign()

In [ ]:
# # Uncomment and run to re-clone repository

# %cd /content
# !rm -rf /content/kg-bayesian-prior
# !git clone https://github.com/ChorokLeeDev/kg-bayesian-prior.git /content/kg-bayesian-prior
# %cd /content/kg-bayesian-prior
# print("Done! Now restart runtime: Runtime -> Restart runtime")